3. 建模

In [1]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # 或者 'Microsoft YaHei'
plt.rcParams['axes.unicode_minus'] = False 

In [2]:
import pandas as pd

# 读取原始数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/专利数据最终文件.csv')

# 分组汇总
result = df.groupby(['year', '企业规模'])['人工智能专利申请数量'].sum().reset_index()

# 重命名列
result.columns = ['年份', '企业规模', '专利总计']

# 保存新文件
result.to_csv('专利建模数据.csv', index=False, encoding='utf-8-sig')

In [3]:
import pandas as pd

# 读取两个文件
df1 = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/全国企业创新调查-主表.csv')
df2 = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/专利建模数据.csv')

# 合并
merged = pd.merge(df1, df2, on=['年份', '企业规模'], how='inner')

# 保存新文件
merged.to_csv('合并专利建模数据.csv', index=False, encoding='utf-8-sig')

In [4]:
import pandas as pd
import statsmodels.api as sm

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/合并专利建模数据.csv')

# 选取自变量和因变量
X = df[['开展创新活动企业数', '创新费用支出合计(亿元)', '开展创新合作企业数', 
        '采取了知识产权保护或相关措施的企业数', '制定创新战略目标的企业数']]
y = df['专利总计']

# 添加常数项
X = sm.add_constant(X)

# 建立回归模型
model = sm.OLS(y, X).fit()

# 输出回归结果
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   专利总计   R-squared:                       0.917
Model:                            OLS   Adj. R-squared:                  0.894
Method:                 Least Squares   F-statistic:                     39.75
Date:                Thu, 29 May 2025   Prob (F-statistic):           4.12e-09
Time:                        06:19:15   Log-Likelihood:                -219.79
No. Observations:                  24   AIC:                             451.6
Df Residuals:                      18   BIC:                             458.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const               5084.8267   1948

/opt/conda/lib/python3.6/site-packages/numpy/core/fromnumeric.py:2389: FutureWarning: Method .ptp is deprecated and will be removed in a future version. Use numpy.ptp instead.
  return ptp(axis=axis, out=out, **kwargs)


In [5]:

# 提取系数表
summary_df = pd.DataFrame(model.summary2().tables[1])
summary_df = summary_df[['Coef.', 'Std.Err.', 't', 'P>|t|']]
summary_df = summary_df.applymap(lambda x: f"{x:.3f}" if isinstance(x, float) else x)

# 提取R方、调整后R方、F检验、Cond. No等
stat_dict = {
    "R方": f"{model.rsquared:.3f}",
    "调整后的R方": f"{model.rsquared_adj:.3f}",
    "F检验": f"{model.fvalue:.3f}",
    "F检验P值": f"{model.f_pvalue:.3f}",
    "Cond. No": f"{model.condition_number:.3f}"
}
stat_df = pd.DataFrame(list(stat_dict.items()), columns=["统计量", "值"])

# 绘制系数表图片
fig1, ax1 = plt.subplots(figsize=(8, 2 + 0.5*len(summary_df)))
ax1.axis('off')
tbl1 = ax1.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    rowLabels=summary_df.index,
    loc='center',
    cellLoc='center',
    rowLoc='center'
)
tbl1.auto_set_font_size(False)
tbl1.set_fontsize(12)
tbl1.scale(1.2, 1.2)
plt.tight_layout()
plt.show()

# 绘制统计量表图片
fig2, ax2 = plt.subplots(figsize=(5, 2 + 0.5*len(stat_df)))
ax2.axis('off')
tbl2 = ax2.table(
    cellText=stat_df.values,
    colLabels=stat_df.columns,
    loc='center',
    cellLoc='center'
)
tbl2.auto_set_font_size(False)
tbl2.set_fontsize(12)
tbl2.scale(1.2, 1.2)
plt.tight_layout()
plt.show()

<Figure size 576x360 with 1 Axes>

<Figure size 360x324 with 1 Axes>